# What a Cut Must Do to Make a Thing

### Congruence, locality, and a predicate with no dynamics

**Dean Kulik** — QuHarmonics — ORCID [0009-0003-3128-8828](https://orcid.org/0009-0003-3128-8828)

Companion notebook. Every number printed in the paper is produced by a cell below.

---

This notebook is **self-contained**. It imports nothing from the companion notebook of
*Two Quotients of One Machine*; every definition it needs is in the single library cell
that follows. Run it top to bottom from a clean kernel. It takes about two minutes.

---

## What this notebook establishes

The previous paper showed that one machine — the $n$-bit ripple adder — carries two
residuals that do not talk to each other: a **state count** that is multiplicative over
the runs of a reader's cut and depends on the reader, and a **rank** that is additive over
pairs of bit positions and does not. This notebook asks a different question of the same
machine.

> A cut produces a quotient. When is that quotient a *thing* rather than a *view*?

The answer that the machine actually supports is a criterion with three clauses, and the
machine supplies one example of each verdict:

| | boundary | compression | closure | verdict |
|---|---|---|---|---|
| **reader cut** $(S,\Phi_S)$ | the read set $S$ | $\Phi$ = the Myhill–Nerode quotient | $U$ local, one-way | **passes** |
| **locality predicate** $K_D$ | the declared squares | $B_A/B_D$ | no non-vacuous transition | **fails** |

Both live on the same machine. That is what makes the criterion non-empty: it is not a
definition that everything satisfies, and it is not one that nothing satisfies.

The results, in order:

- **§1** The cut law $F(S)=\prod_R f(R)$, re-verified here by brute force. Enumeration
  confirms the proof of [1]; it does not replace it.
- **§2** The **full-set ladder**. $F=1$ at *both* ends of a traversal — the empty cut and
  the complete cut are the two stateless cuts. $F=2$ exactly on the bottom-anchored
  intervals and the singleton $\{n-1\}$, the latter a dead end. Hence LSB-first is the
  unique minimal order for $n \ge 3$, under two different cost functionals.
- **§3 Theorem A′.** $\Phi$ is a *congruence*, and the update $U$ is *local*: the next
  signature is computed from the old signature, the cut geometry, the position, and the
  new input pair — never from the assignment. Broken out by all eleven realisable
  (geometry, run type) cases. Part (b), locality, is **verified exhaustively to $n=6$ and
  not proved** — the one headline result here carrying that weaker status. Part (a) turns
  out to be automatic; §4.2 says why.
- **§4 Theorems A″ and A‴.** The update is a function forward and **not a function
  backward**: a reader can close a cut from its state alone, it cannot open one. Then the
  same question off the adder — 52,304 finite transducers — gives a dichotomy: un-reading
  is a function **iff the machine is stateless**. Which reverses the reading. The
  one-wayness is not made by compression at the cut; it is forced by both ends of the
  ladder being stateless, so A″ and Proposition 1 are one fact. The same sweep shows
  Theorem A′(a) is automatic for any machine, leaving all of A′'s content in (b).
- **§5 Theorem B′.** The static side. `declared` is a window predicate; it factors through
  $\mathbb{Z}/2^n \to \mathbb{Z}/2^{n-1}$ and that quotient is exact. The **stabiliser
  law**, proved for every window predicate at every width: $\mathrm{Stab}(D) = \langle
  2^{h+1}\rangle$ where $h$ is the highest bit $D$ effectively reads — so the stabiliser
  always acts trivially on everything $D$ sees. No translation preserves a declared set
  while moving what the predicate looks at.
- **§6** The criterion, and what is proved versus what is only verified.

The two verdicts do not come at the same strength, and the difference is kept visible
throughout: the **failure** is a theorem for all $n$, the **pass** is a finite-width
verification.

---

## 0. Library

One cell, everything. Nothing below this point defines new machinery.

In [3]:
# =====================================================================
#  Self-contained library for "What a Cut Must Do to Make a Thing".
#  Nothing is imported from the companion notebook of the published paper.
#  Everything below is defined here and used by every later cell.
# =====================================================================
import itertools, math, random
import numpy as np

PAIRS = [(0, 0), (0, 1), (1, 0), (1, 1)]      # the four possible (a_q, b_q)
P = 1000003                                    # prime for rank computations

# --------------------------------------------------------------- the cut
def runs_of(S, n):
    """The maximal runs of consecutive read positions in S, as (lo, hi) inclusive."""
    out = []
    for p in sorted(S):
        if out and out[-1][1] == p - 1:
            out[-1][1] = p
        else:
            out.append([p, p])
    return [tuple(r) for r in out]

def f_run(l, r, n):
    """The per-run factor of the cut law."""
    L = r - l + 1
    bot, top = (l == 0), (r == n - 1)
    if bot and top: return 1          # full run: nothing owed, nothing retained
    if bot:         return 2          # bottom-anchored: the carry-out, one bit
    if top:         return L + 1      # top-touching: the deferred-output ledger
    return 2 * L + 1                  # interior: ledger x carry-out

def F(S, n):
    """The cut law:  F(S) = prod over maximal runs R of S of f(R)."""
    v = 1
    for (l, r) in runs_of(S, n):
        v *= f_run(l, r, n)
    return v

# --------------------------------------------------- the machine and its quotient
def mn_class(S, n, data):
    """Myhill-Nerode class of the assignment `data` on the read set S.

    The machine is the n-bit ripple adder.  Its outputs are the n sum bits of
    (a+b) mod 2^n.  A sum bit is emitted the moment no unread position can still
    change it.  Two assignments are the same state iff every completion yields the
    same not-yet-emitted outputs, so the class is the map

        completion  |-->  the pending output coordinates and their values.
    """
    S = sorted(S)
    U = [p for p in range(n) if p not in S]
    outs = []
    for uv in itertools.product(PAIRS, repeat=len(U)):
        a = b = 0
        for p in S:
            x, y = data[p]; a |= x << p; b |= y << p
        for p, (x, y) in zip(U, uv):
            a |= x << p; b |= y << p
        s = (a + b) % (1 << n)
        outs.append(tuple((s >> i) & 1 for i in range(n)))
    pending = tuple(i for i in range(n) if len({o[i] for o in outs}) > 1)
    return (pending, tuple(tuple(o[i] for i in pending) for o in outs))

def signature(S, n, data):
    """Phi: one tuple per maximal run.  The claim is that this IS the MN quotient.

    Per position q the pair (a_q, b_q) is  kill (0,0) | set (1,1) | transparent (0,1),(1,0).
      full run      ("full",)                nothing survives
      bottom run    ("bot", c)               the actual outgoing carry
      transparent   ("trans",)               the whole run passes its carry through
      top run       ("top", d)               depth of the first non-transparent position
      interior run  ("int", d, cout)         that depth, and the outgoing carry
    """
    sig = []
    for (l, r) in runs_of(set(S), n):
        L = r - l + 1
        if l == 0 and r == n - 1:
            sig.append(("full",)); continue
        if l == 0:
            c = 0
            for q in range(l, r + 1):
                a, b = data[q]; c = (a & b) | (c & (a ^ b))
            sig.append(("bot", c)); continue
        d = None
        for t in range(L):
            a, b = data[l + t]
            if (a ^ b) == 0:
                d = t + 1; break
        if d is None:
            sig.append(("trans",)); continue
        if r == n - 1:
            sig.append(("top", d)); continue
        cout = None
        for t in range(L - 1, -1, -1):
            a, b = data[l + t]
            if (a ^ b) == 0:
                cout = a & b; break
        sig.append(("int", d, cout))
    return tuple(sig)

# ------------------------------------------------------------- the local update U
def transfer(sig):
    """How a run passes a carry through: 'id', ('const', v), or None (top: unobservable)."""
    if sig[0] == "trans": return "id"
    if sig[0] == "bot":   return ("const", sig[1])
    if sig[0] == "int":   return ("const", sig[2])
    return None

def depth(sig):
    if sig[0] == "trans": return None
    if sig[0] in ("int", "top"): return sig[1]
    return "bot"

def geometry(S, n, p):
    """open | extend | merge -- the three ways a new position can meet the cut."""
    rr = runs_of(set(S), n)
    lo = any(r == p - 1 for l, r in rr)
    hi = any(l == p + 1 for l, r in rr)
    if lo and hi: return "merge"
    if lo or hi:  return "extend"
    return "open"

def update(sig, S, n, p, pair):
    """U: the next signature from the OLD signature, the cut geometry, the new position
    and the new input pair ONLY.  The underlying assignment is never consulted."""
    rr = runs_of(set(S), n)
    lo = next((t for t, (l, r) in enumerate(rr) if r == p - 1), None)
    hi = next((t for t, (l, r) in enumerate(rr) if l == p + 1), None)
    a, b = pair
    ptype = "trans" if (a ^ b) else ("set" if (a & b) else "kill")
    parts = [(rr[t], sig[t]) for t in range(len(rr))]
    touched = [t for t in (lo, hi) if t is not None]
    keep = [parts[t] for t in range(len(parts)) if t not in touched]
    l_new = rr[lo][0] if lo is not None else p
    r_new = rr[hi][1] if hi is not None else p
    Llow = (rr[lo][1] - rr[lo][0] + 1) if lo is not None else 0
    bot_new, top_new = (l_new == 0), (r_new == n - 1)
    slow = sig[lo] if lo is not None else None
    shigh = sig[hi] if hi is not None else None
    if bot_new and top_new:
        new = ("full",)
    elif bot_new:
        c = slow[1] if slow is not None else 0
        c = 0 if ptype == "kill" else 1 if ptype == "set" else c
        if shigh is not None:
            tr = transfer(shigh)
            c = c if tr == "id" else tr[1]
        new = ("bot", c)
    else:
        dlow = depth(slow) if slow is not None else None
        if dlow is not None:
            d = dlow
        elif ptype != "trans":
            d = Llow + 1
        else:
            dh = depth(shigh) if shigh is not None else None
            d = None if dh is None else Llow + 1 + dh
        if d is None:
            new = ("trans",)
        elif top_new:
            new = ("top", d)
        else:
            trh = transfer(shigh) if shigh is not None else "id"
            if shigh is not None and trh != "id":
                cout = trh[1]
            elif ptype != "trans":
                cout = 1 if ptype == "set" else 0
            else:
                trl = transfer(slow) if slow is not None else "id"
                cout = trl[1] if (slow is not None and trl != "id") else None
            new = ("int", d, cout)
    out = keep + [((l_new, r_new), new)]
    out.sort(key=lambda z: z[0][0])
    return tuple(z[1] for z in out)

# ----------------------------------------------------------------- the static side
def arena(n):
    """Cayley graph of Z/2^n with generators 2^0 .. 2^(n-1).  2^(n-1) is an involution,
    so the graph is simple."""
    N = 1 << n
    eidx, elist = {}, []
    for x in range(N):
        for i in range(n):
            y = (x + (1 << i)) % N
            key = (min(x, y), max(x, y))
            if key not in eidx:
                eidx[key] = len(elist); elist.append((x, i, y))
    return N, eidx, elist

def sgn_edge(eidx, x, i, n):
    y = (x + (1 << i)) % (1 << n)
    return eidx[(min(x, y), max(x, y))], (1 if x < y else -1)

def declared(x, i, j, n):
    """The locality predicate: the moves +2^i and +2^j at x flip disjoint bit sets."""
    N = 1 << n
    return ((x ^ ((x + (1 << i)) % N)) & (x ^ ((x + (1 << j)) % N))) == 0

def window_rule(x, i, j, n):
    """Proposed closed form for `declared`: bits i..j-1 of x are NOT all 1."""
    mask = ((1 << (j - i)) - 1) << i
    return (x & mask) != mask

def rank_modp(rows, ncols):
    if not rows: return 0
    M = np.zeros((len(rows), ncols), dtype=np.int64)
    for a, r in enumerate(rows):
        for c, v in r.items(): M[a, c] = v % P
    r = 0
    for c in range(ncols):
        nz = np.nonzero(M[r:, c])[0]
        if nz.size == 0: continue
        piv = r + nz[0]; M[[r, piv]] = M[[piv, r]]
        M[r] = (M[r] * pow(int(M[r, c]), P - 2, P)) % P
        below = np.nonzero(M[r + 1:, c])[0]
        if below.size:
            idx = below + r + 1
            M[idx] = (M[idx] - np.outer(M[idx, c], M[r])) % P
        r += 1
        if r == M.shape[0]: break
    return r

def b1_of(n, pred):
    """First Betti number of the 2-complex built from the squares `pred` declares."""
    N, eidx, elist = arena(n)
    rows = []
    for x in range(N):
        for i, j in itertools.combinations(range(n), 2):
            if not pred(x, i, j): continue
            r = {}
            for (base, gen, s) in ((x, i, +1), ((x + (1 << i)) % N, j, +1),
                                   (x, j, -1), ((x + (1 << j)) % N, i, -1)):
                k, sg = sgn_edge(eidx, base, gen, n); r[k] = r.get(k, 0) + s * sg
            rows.append(r)
    return len(elist) - N + 1 - rank_modp(rows, len(elist)), len(rows)

def stabiliser(n, pred, pairs=None):
    """Translations v with pred(x+v, i, j) = pred(x, i, j) for every x and pair."""
    N = 1 << n
    pairs = pairs or list(itertools.combinations(range(n), 2))
    return [v for v in range(N)
            if all(pred((x + v) % N, i, j) == pred(x, i, j) for x in range(N) for i, j in pairs)]

def effective_top(n, pred, pairs=None):
    """The highest bit t whose flip changes some declared value; -1 if none."""
    pairs = pairs or list(itertools.combinations(range(n), 2))
    h = -1
    for t in range(n):
        if any(pred(x ^ (1 << t), i, j) != pred(x, i, j)
               for x in range(1 << n) for i, j in pairs):
            h = t
    return h

print("library loaded")

library loaded


---

## 1. The cut law

A **cut** is a set $S \subseteq \{0,\dots,n-1\}$ of positions whose input pairs
$(a_q,b_q)$ have been read. It is a *set*, not a history: the order in which those
positions were read leaves no trace in anything below. What remains of the route is its
boundary geometry, and that is all the machine keeps.

For a maximal run $R = [l,r] \subseteq S$ of length $L = r-l+1$:

```
    full        l = 0, r = n-1     f = 1          nothing owed, nothing retained
    bottom      l = 0, r < n-1     f = 2          the outgoing carry, one bit
    top         l > 0, r = n-1     f = L + 1      the deferred-output ledger
    interior    0 < l, r < n-1     f = 2L + 1     ledger x carry-out
```

and

$$F(S) \;=\; \prod_{R \text{ a maximal run of } S} f(R).$$

The product rather than a sum is forced by a single unread position between two runs: it
can kill a carry, set one, or transmit one, and those three behaviours decouple the runs
on either side of it. The factors are what survives that decoupling.

The first cell checks two things at once, over **every** cut at each width: that the
brute-force Myhill–Nerode count equals $F(S)$, and that $\Phi$ never splits an
MN class. This re-*verifies* the result of [1] rather than re-deriving it — enumeration
confirms a proof, it does not stand in for one.

In [5]:
print("F(S) = prod_R f(R), over EVERY read set, brute force\n")
for n in (2, 3, 4, 5, 6):
    bad = sets = refine = 0
    for k in range(n + 1):
        for S in itertools.combinations(range(n), k):
            S = list(S); sets += 1
            cls, byphi = set(), {}
            for kv in itertools.product(PAIRS, repeat=k):
                data = dict(zip(S, kv))
                m = mn_class(S, n, data)
                cls.add(m)
                byphi.setdefault(signature(S, n, data), set()).add(m)
            bad    += (len(cls) != F(S, n))
            refine += sum(len(v) != 1 for v in byphi.values())
    print(f"  n={n}: {sets:3d} read sets | {bad} where |MN| != prod f(R) | "
          f"{refine} Phi-classes that split   {'OK' if bad == 0 and refine == 0 else 'FAIL'}")

F(S) = prod_R f(R), over EVERY read set, brute force

  n=2:   4 read sets | 0 where |MN| != prod f(R) | 0 Phi-classes that split   OK
  n=3:   8 read sets | 0 where |MN| != prod f(R) | 0 Phi-classes that split   OK
  n=4:  16 read sets | 0 where |MN| != prod f(R) | 0 Phi-classes that split   OK
  n=5:  32 read sets | 0 where |MN| != prod f(R) | 0 Phi-classes that split   OK
  n=6:  64 read sets | 0 where |MN| != prod f(R) | 0 Phi-classes that split   OK


Refinement alone is weak — the identity map refines everything. The reverse direction
is the one that matters: $\Phi$ must not be *finer* than the quotient either, or it would
be retaining distinctions the future cannot use. Both counts must equal $F(S)$.

In [7]:
print("Phi is EXACTLY the MN quotient:  |image Phi| = |MN classes| = F(S)\n")
for n in (2, 3, 4, 5, 6):
    bad = 0
    for k in range(n + 1):
        for S in itertools.combinations(range(n), k):
            S = list(S)
            phis, mns = set(), set()
            for kv in itertools.product(PAIRS, repeat=k):
                data = dict(zip(S, kv))
                phis.add(signature(S, n, data)); mns.add(mn_class(S, n, data))
            bad += not (len(phis) == len(mns) == F(S, n))
    print(f"  n={n}: {bad} read sets where the three counts disagree   "
          f"{'OK' if bad == 0 else 'FAIL'}")

Phi is EXACTLY the MN quotient:  |image Phi| = |MN classes| = F(S)

  n=2: 0 read sets where the three counts disagree   OK
  n=3: 0 read sets where the three counts disagree   OK
  n=4: 0 read sets where the three counts disagree   OK
  n=5: 0 read sets where the three counts disagree   OK
  n=6: 0 read sets where the three counts disagree   OK


---

## 2. The full-set ladder

Now the question this paper is built around. $F$ counts what a cut must retain. Read that
count as a statement about *existence*: a cut with nothing to retain has no state, and a
thing with no state is not a thing — it is a view.

### 2.1 Both ends are stateless

The first assertion I wrote here was "$F(S)=1$ if and only if $S=[0,n-1]$". It is false,
and the failure is the point of the section. The empty cut has $F=1$ as well — the empty
product. So:

$$F(S) = 1 \iff S = \varnothing \ \text{ or } \ S = [0,n-1].$$

Nothing read, and everything read. One has no continuation debt because nothing has
happened yet; the other has none because nothing remains. The state count alone cannot
tell them apart. **The cut can.** That is the sharpest form of the claim that the object
is the cut and not the count: two cuts with identical quotients, and they are opposite
conditions.

In [9]:
print("F(S) = 1  <=>  S = {} or S = [0, n-1]\n")
for n in range(2, 9):
    ones = [tuple(S) for k in range(n + 1) for S in itertools.combinations(range(n), k)
            if F(list(S), n) == 1]
    ok = ones == [(), tuple(range(n))]
    print(f"  n={n}: F=1 on {len(ones)} cut(s): {ones}   {'OK' if ok else 'FAIL'}")

F(S) = 1  <=>  S = {} or S = [0, n-1]

  n=2: F=1 on 2 cut(s): [(), (0, 1)]   OK
  n=3: F=1 on 2 cut(s): [(), (0, 1, 2)]   OK
  n=4: F=1 on 2 cut(s): [(), (0, 1, 2, 3)]   OK
  n=5: F=1 on 2 cut(s): [(), (0, 1, 2, 3, 4)]   OK
  n=6: F=1 on 2 cut(s): [(), (0, 1, 2, 3, 4, 5)]   OK
  n=7: F=1 on 2 cut(s): [(), (0, 1, 2, 3, 4, 5, 6)]   OK
  n=8: F=1 on 2 cut(s): [(), (0, 1, 2, 3, 4, 5, 6, 7)]   OK


### 2.2 The minimal thing

$F=2$ is the smallest non-trivial state: one retained bit. Which cuts have it?

In [11]:
print("F(S) = 2  <=>  S = [0,r] with r < n-1,  or  S = {n-1}\n")
for n in range(2, 9):
    twos = sorted(tuple(S) for k in range(n + 1)
                  for S in itertools.combinations(range(n), k) if F(list(S), n) == 2)
    pred = sorted([tuple(range(r + 1)) for r in range(n - 1)] + [(n - 1,)])
    print(f"  n={n}: {len(twos)} cuts, {'matches' if twos == pred else 'DIFFERS FROM'} "
          f"the closed form")
    if n <= 4: print(f"        {twos}")

F(S) = 2  <=>  S = [0,r] with r < n-1,  or  S = {n-1}

  n=2: 2 cuts, matches the closed form
        [(0,), (1,)]
  n=3: 3 cuts, matches the closed form
        [(0,), (0, 1), (2,)]
  n=4: 4 cuts, matches the closed form
        [(0,), (0, 1), (0, 1, 2), (3,)]
  n=5: 5 cuts, matches the closed form
  n=6: 6 cuts, matches the closed form
  n=7: 7 cuts, matches the closed form
  n=8: 8 cuts, matches the closed form


Two families, and they behave very differently. The bottom-anchored intervals $[0,r]$
extend: adding $r+1$ gives $[0,r+1]$, still $F=2$. The singleton $\{n-1\}$ does not.

In [13]:
print("{n-1} is a dead end: no F=2 cut properly contains it\n")
for n in range(2, 9):
    dead = all(F(list(S), n) != 2
               for k in range(2, n + 1) for S in itertools.combinations(range(n), k)
               if (n - 1) in S)
    print(f"  n={n}: {'dead end  OK' if dead else 'FAIL'}")

{n-1} is a dead end: no F=2 cut properly contains it

  n=2: dead end  OK
  n=3: dead end  OK
  n=4: dead end  OK
  n=5: dead end  OK
  n=6: dead end  OK
  n=7: dead end  OK
  n=8: dead end  OK


So there is exactly one way to walk from $\varnothing$ to $[0,n-1]$ holding $F \le 2$
the whole way: take position $0$ first, because $\{n-1\}$ leads nowhere, and then stay
inside the bottom-anchored family, which forces $[0,t-1]$ at step $t$. That is LSB-first.

The carry bit is not a fact about addition. It is a fact about bottom-anchored cuts.

Two cost functionals, minimised over all $n!$ orders. The min–max is the peak state count;
the sum of $\log_2 F$ is the integrated width of the retained state over the traversal.
They need not have the same minimiser — that is a question, not an assumption.

In [15]:
print(f"{'n':>2} {'min max F':>10} {'#argmin':>8}   {'min sum log2 F':>15} {'#argmin':>8}"
      f"   {'same order?':>12}  {'LSB optimal?':>13}")
for n in range(2, 9):
    best_mx = best_sm = None; arg_mx, arg_sm = [], []
    for o in itertools.permutations(range(n)):
        prof = [F(set(o[:t]), n) for t in range(n + 1)]
        mx, sm = max(prof), sum(math.log2(v) for v in prof)
        if best_mx is None or mx < best_mx:        best_mx, arg_mx = mx, [o]
        elif mx == best_mx:                        arg_mx.append(o)
        if best_sm is None or sm < best_sm - 1e-9: best_sm, arg_sm = sm, [o]
        elif abs(sm - best_sm) < 1e-9:             arg_sm.append(o)
    lsb = tuple(range(n))
    print(f"{n:>2} {best_mx:>10} {len(arg_mx):>8}   {best_sm:>15.4f} {len(arg_sm):>8}"
          f"   {str(set(arg_mx) == set(arg_sm)):>12}  "
          f"{str(lsb in arg_mx and lsb in arg_sm):>13}")

 n  min max F  #argmin    min sum log2 F  #argmin    same order?   LSB optimal?
 2          2        2            1.0000        2           True           True
 3          2        1            2.0000        1           True           True
 4          2        1            3.0000        1           True           True
 5          2        1            4.0000        1           True           True
 6          2        1            5.0000        1           True           True
 7          2        1            6.0000        1           True           True
 8          2        1            7.0000        1           True           True


Both functionals, same unique minimiser, for every $n \ge 3$; at $n=2$ both orders tie,
which is the one exception and it is visible in the `#argmin` column rather than asserted
away. The minimum of $\sum \log_2 F$ is exactly $n-1$, because the profile is
$1,2,2,\dots,2,1$: one retained bit at each of the $n-1$ interior cut positions, and none
at the two ends.

### 2.3 Cuts that are not full

Every cut with an interior run picks up an odd factor $2L+1$. An odd factor greater than
one is never a power of two, so such a state does not decompose into bits — there is no
basis in which the reader is holding $k$ independent flags.

In [17]:
print("no cut with an interior run has a power-of-two state count\n")
for n in (5, 6, 7, 8):
    inter = bad = 0
    for k in range(n + 1):
        for S in itertools.combinations(range(n), k):
            if any(l > 0 and r < n - 1 for (l, r) in runs_of(set(S), n)):
                inter += 1
                v = F(list(S), n)
                bad += (v & (v - 1)) == 0
    print(f"  n={n}: {inter:4d} cuts with an interior run | {bad} a power of two   "
          f"{'OK' if bad == 0 else 'FAIL'}")

no cut with an interior run has a power-of-two state count

  n=5:   16 cuts with an interior run | 0 a power of two   OK
  n=6:   42 cuts with an interior run | 0 a power of two   OK
  n=7:   99 cuts with an interior run | 0 a power of two   OK
  n=8:  219 cuts with an interior run | 0 a power of two   OK


That is the quantitative content of "the boundary is not closed". A cut with a gap is
not a slightly worse object; it is holding a ledger whose size is not a number of bits at
all. Closure is the condition under which the retained state becomes a *register*.

### 2.4 What survives a change of base

Everything above was computed in base 2 and none of it said so. So: which parts are facts
about cuts and which about binary? Run the same machine in base $b$ — digits in
$\{0,\dots,b-1\}$, $s_q = (a_q+b_q+c_q) \bmod b$, $c_{q+1} = \lfloor(a_q+b_q+c_q)/b\rfloor$.
Two readings, pointing opposite ways:

| reading | prediction |
|---|---|
| forward | the cut law is about addition, so it should look the same in every base |
| reverse | the factors count what a reader *owes*, and that depends on how large a digit can be, so they should grow with $b$ |

Each is right about part of it, and which part is the answer.

In [19]:
print("the base-b reader: is it still a product over runs, and with what factors?\n")

def mn_b(n, b, S, data):
    S = sorted(S); U = [p for p in range(n) if p not in S]; outs = []
    for uv in itertools.product(range(2*b - 1), repeat=len(U)):
        tot = dict(data); tot.update(dict(zip(U, uv)))
        c = 0; s = []
        for q in range(n):
            t = tot[q] + c; s.append(t % b); c = t // b
        outs.append(tuple(s))
    pend = tuple(i for i in range(n) if len({o[i] for o in outs}) > 1)
    return (pend, tuple(tuple(o[i] for i in pend) for o in outs))

def Fb(n, b, S):
    S = sorted(S)
    return len({mn_b(n, b, S, dict(zip(S, kv)))
                for kv in itertools.product(range(2*b - 1), repeat=len(S))})

for b in (2, 3, 4):
    for n in (3, 4):
        single = {(l, r): Fb(n, b, list(range(l, r+1)))
                  for l in range(n) for r in range(l, n)}
        bad = 0
        for k in range(n + 1):
            for S in itertools.combinations(range(n), k):
                pred = 1
                for R in runs_of(set(S), n): pred *= single[R]
                bad += Fb(n, b, list(S)) != pred
        print(f"  b={b} n={n}: product law {'holds' if bad == 0 else 'FAILS'}  ({bad} exceptions)")

print("\nthe per-run factors against  f_full=1 | f_bot=2 | f_top=(b-1)L+1 | f_int=2(b-1)L+1\n")
bad = tot = 0
for b in (2, 3, 4, 5):
    for n in (3, 4):
        if (2*b - 1) ** n > 30000: continue
        for l in range(n):
            for r in range(l, n):
                L = r - l + 1
                kind = ("full" if (l == 0 and r == n-1) else "bottom" if l == 0
                        else "top" if r == n-1 else "interior")
                want = {"full": 1, "bottom": 2, "top": (b-1)*L + 1,
                        "interior": 2*(b-1)*L + 1}[kind]
                got = Fb(n, b, list(range(l, r+1)))
                tot += 1; bad += got != want
print(f"  {tot} single-run cuts over b=2..5: {bad} mismatches   "
      f"{'OK' if bad == 0 else 'FAIL'}")

print(f"\n  {'quantity':<38} {'b=2':>6}{'b=3':>6}{'b=4':>6}{'b=5':>6}")
for name, fn in (("f(full run)", lambda b: 1),
                 ("f(bottom-anchored run), any L", lambda b: 2),
                 ("f(top run), L = 2", lambda b: (b-1)*2 + 1),
                 ("f(interior run), L = 2", lambda b: 2*(b-1)*2 + 1)):
    print(f"  {name:<38}" + "".join(f"{fn(b):>6}" for b in (2, 3, 4, 5)))

print("\nand the reader-side n-1, by base\n")
for n in (3, 4):
    line = f"  n={n}: "
    for b in (2, 3, 4):
        if (2*b - 1) ** n > 30000: continue
        best = min(sum(math.log2(Fb(n, b, list(o[:t]))) for t in range(n + 1))
                   for o in itertools.permutations(range(n)))
        line += f" b={b}: {best:.4f}  "
    print(line + f"   (n-1 = {n-1})")

the base-b reader: is it still a product over runs, and with what factors?

  b=2 n=3: product law holds  (0 exceptions)
  b=2 n=4: product law holds  (0 exceptions)
  b=3 n=3: product law holds  (0 exceptions)
  b=3 n=4: product law holds  (0 exceptions)
  b=4 n=3: product law holds  (0 exceptions)
  b=4 n=4: product law holds  (0 exceptions)

the per-run factors against  f_full=1 | f_bot=2 | f_top=(b-1)L+1 | f_int=2(b-1)L+1

  64 single-run cuts over b=2..5: 0 mismatches   OK

  quantity                                  b=2   b=3   b=4   b=5
  f(full run)                                1     1     1     1
  f(bottom-anchored run), any L              2     2     2     2
  f(top run), L = 2                          3     5     7     9
  f(interior run), L = 2                     5     9    13    17

and the reader-side n-1, by base

  n=3:  b=2: 2.0000   b=3: 2.0000   b=4: 2.0000     (n-1 = 2)
  n=4:  b=2: 3.0000   b=3: 3.0000   b=4: 3.0000     (n-1 = 3)


  64 single-run cuts over b=2..5: 0 mismatches   OK

  quantity                                  b=2   b=3   b=4   b=5
  f(full run)                                1     1     1     1
  f(bottom-anchored run), any L              2     2     2     2
  f(top run), L = 2                          3     5     7     9
  f(interior run), L = 2                     5     9    13    17

and the reader-side n-1, by base

  n=3:  b=2: 2.0000   b=3: 2.0000   b=4: 2.0000     (n-1 = 2)


  n=4:  b=2: 3.0000   b=3: 3.0000   b=4: 3.0000     (n-1 = 3)


So the base-2 formulas $L+1$ and $2L+1$ are the $b=2$ cases of $(b-1)L+1$ and
$2(b-1)L+1$. **Invariant:** the decomposition into runs, $f(	ext{full})=1$,
$f(	ext{bottom})=2$, the factor of two between top and interior, the oddness of the
interior factor, and $\min \sum \log_2 F = n-1$. **Not invariant:** the ledger sizes,
which scale with $b-1$ — as they must, since the ledger records *which digit value* is
owed and there are more of them.

One factor refuses to move, and it is the important one: $f=2$ for a bottom-anchored run,
in every base. Two-operand addition has a carry in $\{0,1\}$ whatever the base, because
$\lfloor (a+b+c)/b 
floor$ with $a,b \le b-1$ and $c \le 1$ never reaches 2.

$$	ext{The carry bit is not a fact about binary either — it is a fact about bottom-anchored cuts of two-operand addition.}$$

Corollary 4 carries over unchanged: its argument used only Propositions 1 and 2, which
depend on $f(	ext{full})=1$ and $f(	ext{bottom})=2$, both base-invariant.

---

## 3. Theorem A′: the reader is a machine

The previous paper proved that $\Phi$ is a sufficient statistic: the future depends on the
past only through $\Phi$. That is one half of what "state" means. The other half is that
$\Phi$ must be *carried forward* — computed from itself, not recomputed from the history
it summarises. A summary that has to be rebuilt from the whole record each step is a
description of the system, not a state of it.

**Theorem A′.** For every $n$, every cut $S$, every position $p \notin S$ and every input
pair $u$:

&nbsp;&nbsp;**(a) Congruence.**
$\ \Phi_S(\alpha) = \Phi_S(\beta) \implies \Phi_{S \cup \{p\}}(\alpha[p \mapsto u]) =
\Phi_{S \cup \{p\}}(\beta[p \mapsto u]).$

&nbsp;&nbsp;**(b) Locality.** There is a map $U$ with
$\ \Phi_{S \cup \{p\}}(\alpha[p \mapsto u]) = U(\Phi_S(\alpha),\, S,\, p,\, u)$
which reads only the at most two run signatures adjacent to $p$, the geometry of $p$
relative to $S$, and the transfer type of $u$ — never the assignment $\alpha$.

Together: $(S, \Phi_S) \xrightarrow{\ (p,u)\ } (S \cup \{p\}, \Phi_{S\cup\{p\}})$ is a
transition system.

Note what (b) makes explicit. The state is the pair $(S, \Phi_S)$, not $\Phi_S$ alone:
$U$ needs the cut geometry to know whether the new position opens a run, extends one, or
closes a gap between two. **Position is part of machine state.** A reader that does not
know where it is cannot carry its own quotient forward.

In [22]:
print("(a) congruence: equal state + same (position, pair) -> equal next state\n")
for n in (3, 4, 5, 6):
    tot = bad = 0
    for k in range(n):
        for S in itertools.combinations(range(n), k):
            S = sorted(S); byphi = {}
            for kv in itertools.product(PAIRS, repeat=len(S)):
                byphi.setdefault(signature(S, n, dict(zip(S, kv))), []).append(dict(zip(S, kv)))
            for p in range(n):
                if p in S: continue
                for pair in PAIRS:
                    for _, members in byphi.items():
                        outs = {signature(sorted(S + [p]), n, {**d, p: pair}) for d in members}
                        tot += 1; bad += len(outs) > 1
    print(f"  n={n}: {tot:6d} (state, position, pair) triples | {bad} undetermined   "
          f"{'OK' if bad == 0 else 'FAIL'}")

(a) congruence: equal state + same (position, pair) -> equal next state

  n=3:    104 (state, position, pair) triples | 0 undetermined   OK
  n=4:    408 (state, position, pair) triples | 0 undetermined   OK
  n=5:   1512 (state, position, pair) triples | 0 undetermined   OK
  n=6:   5400 (state, position, pair) triples | 0 undetermined   OK


Now locality, and not as a single aggregate number. A new position meets the cut in
exactly three ways — it **opens** a run of its own, **extends** an existing run at one end,
or **merges** two runs by closing the one-position gap between them. (A fourth case
sometimes written down, "both neighbours in the same run", cannot occur: $p \notin S$
separates them, so if both neighbours are present they belong to distinct runs.) Crossed
with what the resulting run becomes — bottom, top, interior or full — that gives the table
below. The merge cases are the load-bearing ones: that is where two independent ledgers
have to be composed into one, and where a local rule could fail without the aggregate
count noticing.

In [24]:
print("(b) locality of U, by (geometry of p, type of the resulting run)\n")
for n in (3, 4, 5, 6):
    tot = bad = 0; bycase = {}
    for k in range(n):
        for S in itertools.combinations(range(n), k):
            S = sorted(S)
            for p in range(n):
                if p in S: continue
                rr_new = runs_of(set(S) | {p}, n)
                l, r = next((l, r) for (l, r) in rr_new if l <= p <= r)
                kind = ("full" if (l == 0 and r == n - 1) else "bottom" if l == 0
                        else "top" if r == n - 1 else "interior")
                key = (geometry(S, n, p), kind)
                for kv in itertools.product(PAIRS, repeat=len(S)):
                    data = dict(zip(S, kv)); sig = signature(S, n, data)
                    for pair in PAIRS:
                        wrong = update(sig, S, n, p, pair) != signature(
                            sorted(S + [p]), n, {**data, p: pair})
                        tot += 1; bad += wrong
                        t, w = bycase.get(key, (0, 0)); bycase[key] = (t + 1, w + wrong)
    print(f"  n={n}: {tot:7d} updates | {bad} wrong   {'OK' if bad == 0 else 'FAIL'}")
    if n == 6:
        print(f"\n  the eleven cases at n=6:")
        print(f"    {'geometry':<9} {'run becomes':<11} {'updates':>9} {'wrong':>6}")
        for key in sorted(bycase):
            t, w = bycase[key]
            print(f"    {key[0]:<9} {key[1]:<11} {t:>9} {w:>6}")

(b) locality of U, by (geometry of p, type of the resulting run)

  n=3:     300 updates | 0 wrong   OK
  n=4:    2000 updates | 0 wrong   OK
  n=5:   12500 updates | 0 wrong   OK
  n=6:   75000 updates | 0 wrong   OK

  the eleven cases at n=6:
    geometry  run becomes   updates  wrong
    extend    bottom          11808      0
    extend    full             8192      0
    extend    interior         4192      0
    extend    top             11808      0
    merge     bottom           7232      0
    merge     full            16384      0
    merge     interior         1152      0
    merge     top              7232      0
    open      bottom           2500      0
    open      interior         2000      0
    open      top              2500      0


Eleven cases, all covered, none wrong. `open -> full` is absent because a single
isolated position is the whole of $[0,n-1]$ only when $n=1$.

---

## 4. Theorem A″: the cut turns one way

Say it, then reverse it. Forward, the state determines the next state. Backward, does the
state determine the *previous* one? Concretely: if the reader un-reads a position $p$,
is $\Phi_{S \setminus \{p\}}$ a function of $\Phi_S$ at all — never mind local?

This has not been asked before, and the answer is no.

In [26]:
print("is un-reading a function of the state?\n")
first = None
for n in (3, 4, 5, 6):
    tot = bad = 0
    for k in range(1, n + 1):
        for S in itertools.combinations(range(n), k):
            S = sorted(S); byphi = {}
            for kv in itertools.product(PAIRS, repeat=len(S)):
                byphi.setdefault(signature(S, n, dict(zip(S, kv))), []).append(dict(zip(S, kv)))
            for p in S:
                T = [q for q in S if q != p]
                for sig, members in byphi.items():
                    outs = {signature(T, n, {q: d[q] for q in T}) for d in members}
                    tot += 1
                    if len(outs) > 1:
                        bad += 1
                        if first is None: first = (n, S, p, sig, sorted(outs))
    print(f"  n={n}: {tot:6d} (state, forgotten position) pairs | {bad} where the earlier "
          f"state is NOT determined   ({100*bad/tot:.0f}%)")

n, S, p, sig, outs = first
print(f"\n  smallest witness:  n={n}, S={S}, forget p={p}, Phi_S={sig}")
print(f"  compatible earlier states: {outs}")

is un-reading a function of the state?

  n=3:     28 (state, forgotten position) pairs | 9 where the earlier state is NOT determined   (32%)
  n=4:    114 (state, forgotten position) pairs | 37 where the earlier state is NOT determined   (32%)
  n=5:    432 (state, forgotten position) pairs | 141 where the earlier state is NOT determined   (33%)
  n=6:   1566 (state, forgotten position) pairs | 513 where the earlier state is NOT determined   (33%)

  smallest witness:  n=3, S=[0, 1], forget p=0, Phi_S=(('bot', 0),)
  compatible earlier states: [(('int', 1, 0),), (('trans',),)]


The witness generalises to every $n \ge 3$ and needs no search. Take $S=\{0,1\}$ and the
two assignments

$$\alpha:\ (a_0,b_0)=(0,0),\ (a_1,b_1)=(0,1) \qquad
  \beta:\ (a_0,b_0)=(0,0),\ (a_1,b_1)=(0,0).$$

Both kill the carry at position $0$ and emit a carry-out of $0$, so
$\Phi_S(\alpha) = \Phi_S(\beta) = ((\texttt{bot},0),)$. Drop position $0$. The run $[1,1]$
is interior whenever $n \ge 3$, so on the smaller cut $\{1\}$ the two separate: $\alpha$
gives $((\texttt{trans},),)$ and $\beta$ gives $((\texttt{int},1,0),)$. $\square$

The witness is worth reading slowly. The cut $S=\{0,1\}$ is bottom-anchored with
signature $(\texttt{bot},0)$: the reader knows its outgoing carry is $0$ and nothing else,
because everything below has been emitted. Un-read position $0$ and the remaining cut
$\{1\}$ is an interior run, whose state must record whether position $1$ was transparent
or carry-killing. The bottom-anchored state has already thrown that away — correctly, since
with position $0$ in hand the distinction cannot affect anything downstream.

So:

$$\text{a reader can close a cut from its state alone; it cannot open one.}$$

### 4.1 The same statement, off the adder

The obvious reading is that *compression* makes the transition irreversible — discarding
what the future cannot use is what a state is, so one-wayness is manufactured at the cut.
That reading is wrong, and the way to find out is to leave the adder.

Take the family the adder sits inside: left-to-right finite transducers

$$c_0 = 0,\qquad c_{q+1} = g(c_q, u_q),\qquad s_q = f(c_q, u_q)$$

with $u_q$ from an alphabet of size $m$ and $c$ from a state set of size $k$, read by the
same reader. The adder is $k=2$, $m=4$, $g$ = majority, $f$ = parity XOR carry. Sweep
every $g$ and $f$.

Stating this over transducers would be a scope error — the proof never touches one. It
uses two properties of a family of quotients indexed by subsets, so that is the level.

**Theorem A‴ (backward-functionality dichotomy).** Let $I$ be a finite coordinate set with
an alphabet at each coordinate, and for each $S \subseteq I$ let $Q_S$ be a quotient of the
assignments on $S$. Suppose

$$\textbf{(H1)}\ \ \text{restriction } Q_S \to Q_{S\setminus\{p\}} \text{ is onto},
\qquad \textbf{(H2)}\ \ |Q_I| = 1 .$$

If every restriction $Q_S \to Q_{S\setminus\{p\}}$ is a well-defined function on classes,
then $|Q_S| = 1$ for every $S$. Equivalently: if some cut has $|Q_S| > 1$, at least one
un-reading fails to be a function.

*Proof.* By (H1) restriction is onto, so if it is also a function then
$|Q_{S\setminus\{p\}}| \le |Q_S|$. By (H2) the top of the lattice has one class. Removing
coordinates one at a time from $I$ reaches every $S$, and the count never rises. $\square$

The reader of a finite transducer is one instance: (H1) because any assignment on
$S\setminus\{p\}$ extends, (H2) because with everything read nothing is pending.

### 4.2 Which hypothesis is doing the work

Now the hypotheses are separate, so drop one at a time and look for a non-trivial
backward-functional family. **(H1) is not a hypothesis here** — for a quotient of all
partial assignments it is automatic, and over 240 arbitrarily-generated quotients it never
failed. **(H2) is the whole content**: break it and non-trivial backward functionality
returns at once.

The witness is a machine, not a contrivance. Two readers that differ only in whether the
second input is inside the readable set.

In [28]:
print("(H2) broken vs (H2) intact -- the same machine either side of the boundary\n")

def q_family(n, cls):
    # every cut's class set, plus the three checks
    def assigns(S):
        for kv in itertools.product(range(2), repeat=len(S)):
            yield dict(zip(sorted(S), kv))
    cuts = [tuple(sorted(S)) for sz in range(n + 1)
            for S in itertools.combinations(range(n), sz)]
    Q = {S: {cls(S, a) for a in assigns(S)} for S in cuts}
    h1 = all({cls(tuple(q for q in S if q != p), {q: a[q] for q in S if q != p})
              for a in assigns(S)} == Q[tuple(q for q in S if q != p)]
             for S in cuts if S for p in S)
    h2 = len(Q[tuple(range(n))]) == 1
    nt = max(len(v) for v in Q.values()) > 1
    bf = True
    for S in cuts:
        if not S: continue
        by = {}
        for a in assigns(S):
            by.setdefault(cls(S, a), []).append(a)
        for p in S:
            T = tuple(q for q in S if q != p)
            for c, mem in by.items():
                if len({cls(T, {q: a[q] for q in T}) for a in mem}) > 1:
                    bf = False
    return h1, h2, nt, bf

def open_reader(n):
    # output = x_0 XOR ext, with ext OUTSIDE the readable set
    def cls(S, a):
        U = [p for p in range(n) if p not in S]
        rows = set()
        for uv in itertools.product(range(2), repeat=len(U)):
            v = dict(a); v.update(dict(zip(U, uv)))
            rows.add(tuple(v[0] ^ e for e in (0, 1)))
        return tuple(sorted(rows))
    return cls

def closed_reader(n):
    # output = x_0 XOR x_1, both readable
    def cls(S, a):
        U = [p for p in range(n) if p not in S]; outs = []
        for uv in itertools.product(range(2), repeat=len(U)):
            v = dict(a); v.update(dict(zip(U, uv)))
            outs.append((v[0] ^ v[1],))
        pend = tuple(i for i in range(1) if len({o[i] for o in outs}) > 1)
        return (pend, tuple(tuple(o[i] for i in pend) for o in outs))
    return cls

print(f"  {'reader':<34} {'H1':>6} {'H2':>6} {'nontrivial':>11} {'un-read a function':>20}")
for n in (2, 3, 4):
    for lbl, mk in (("open   (ext NOT readable)", open_reader),
                    ("closed (both readable)   ", closed_reader)):
        h1, h2, nt, bf = q_family(n, mk(n))
        print(f"  {lbl} n={n:<18} {str(h1):>6} {str(h2):>6} {str(nt):>11} {str(bf):>20}")
print("\n  one bit moved across the boundary of the readable set, and the one-wayness flips.")

(H2) broken vs (H2) intact -- the same machine either side of the boundary

  reader                                 H1     H2  nontrivial   un-read a function
  open   (ext NOT readable) n=2                    True  False        True                 True
  closed (both readable)    n=2                    True   True        True                False
  open   (ext NOT readable) n=3                    True  False        True                 True
  closed (both readable)    n=3                    True   True        True                False
  open   (ext NOT readable) n=4                    True  False        True                 True
  closed (both readable)    n=4                    True   True        True                False

  one bit moved across the boundary of the readable set, and the one-wayness flips.


$$	ext{A reader closed with respect to what it must report is forced one-way. An open one is not.}$$

The first attempt at this witness failed, and the failure is worth recording: an external
input that does *not* interact with the readable ones is pending at every cut alike, so it
drops out of the class and the family comes back trivial. The escape needs the unreadable
input to couple to a readable one. Openness alone is not enough; openness that couples is.

### 4.3 What the sweep corrects

In [30]:
from collections import defaultdict

def mn_machine(n, S, data, k, m, g, f):
    # MN class of `data` on cut S for the transducer (g, f)
    S = sorted(S); U = [p for p in range(n) if p not in S]; outs = []
    for uv in itertools.product(range(m), repeat=len(U)):
        u = dict(data); u.update(dict(zip(U, uv)))
        c = 0; s = []
        for q in range(n):
            s.append(f[c][u[q]]); c = g[c][u[q]]
        outs.append(tuple(s))
    pend = tuple(i for i in range(n) if len({o[i] for o in outs}) > 1)
    return (pend, tuple(tuple(o[i] for i in pend) for o in outs))

def analyse(n, k, m, g, f):
    by = {}
    for sz in range(n + 1):
        for S in itertools.combinations(range(n), sz):
            d = defaultdict(list)
            for kv in itertools.product(range(m), repeat=len(S)):
                a = dict(zip(S, kv)); d[mn_machine(n, S, a, k, m, g, f)].append(a)
            by[S] = d
    mx = max(len(v) for v in by.values())
    fwd = bwd = True
    for S, d in by.items():
        for p in range(n):
            if p in S: continue
            T = tuple(sorted(S + (p,)))
            for u in range(m):
                for _, mem in d.items():
                    if len({mn_machine(n, T, {**a, p: u}, k, m, g, f) for a in mem}) > 1:
                        fwd = False
        for p in S:
            T = tuple(q for q in S if q != p)
            for _, mem in d.items():
                if len({mn_machine(n, T, {q: a[q] for q in T}, k, m, g, f) for a in mem}) > 1:
                    bwd = False
    return (mx > 1), fwd, bwd, mx

def maps(k, m, rng):
    for flat in itertools.product(range(rng), repeat=k * m):
        yield [list(flat[i*m:(i+1)*m]) for i in range(k)]

# control: the adder lives in this family
PR = [(0,0),(0,1),(1,0),(1,1)]
g_add = [[(a & b) | (c & (a ^ b)) for (a,b) in PR] for c in (0,1)]
f_add = [[(a ^ b) ^ c              for (a,b) in PR] for c in (0,1)]
print("control -- the adder as a transducer\n")
for n in (3, 4, 5):
    nt, fw, bw, mx = analyse(n, 2, 4, g_add, f_add)
    print(f"  n={n}: nontrivial={nt}  forward={fw}  backward={bw}  max states={mx}   "
          f"{'as Theorem A-double-prime says' if (nt and fw and not bw) else 'UNEXPECTED'}")

print("\nthe sweep: backward-functional <=> stateless?\n")
grand = 0
for k, m, rng, lbl in ((2,2,2,"k=2 m=2 binary"), (2,3,2,"k=2 m=3 binary"),
                       (2,2,3,"k=2 m=2 ternary")):
    cnt = mism = nofwd = 0
    for g in maps(k, m, k):
        for f in maps(k, m, rng):
            cnt += 1
            nt, fw, bw, mx = analyse(4, k, m, g, f)
            mism += (bw != (not nt)); nofwd += (not fw)
    grand += cnt
    print(f"  {lbl:<16} {cnt:6d} machines | forward failures {nofwd} | "
          f"backward != stateless in {mism}   {'OK' if mism == nofwd == 0 else 'FAIL'}")
print(f"\n  {grand} machines at n=4, no exceptions in either direction.")
print("  (k=3, m=2 adds 46,656 more, also with none; omitted here for runtime.)")

control -- the adder as a transducer

  n=3: nontrivial=True  forward=True  backward=False  max states=4   as Theorem A-double-prime says
  n=4: nontrivial=True  forward=True  backward=False  max states=6   as Theorem A-double-prime says
  n=5: nontrivial=True  forward=True  backward=False  max states=12   as Theorem A-double-prime says

the sweep: backward-functional <=> stateless?

  k=2 m=2 binary      256 machines | forward failures 0 | backward != stateless in 0   OK
  k=2 m=3 binary     4096 machines | forward failures 0 | backward != stateless in 0   OK
  k=2 m=2 ternary    1296 machines | forward failures 0 | backward != stateless in 0   OK

  5648 machines at n=4, no exceptions in either direction.
  (k=3, m=2 adds 46,656 more, also with none; omitted here for runtime.)


  k=2 m=2 binary      256 machines | forward failures 0 | backward != stateless in 0   OK


  k=2 m=3 binary     4096 machines | forward failures 0 | backward != stateless in 0   OK


  k=2 m=2 ternary    1296 machines | forward failures 0 | backward != stateless in 0   OK

  5648 machines at n=4, no exceptions in either direction.
  (k=3, m=2 adds 46,656 more, also with none; omitted here for runtime.)


In [31]:
print("the two facts the proof actually uses\n")
def classes(n, S, k, m, g, f):
    d = defaultdict(list)
    for kv in itertools.product(range(m), repeat=len(S)):
        a = dict(zip(sorted(S), kv)); d[mn_machine(n, sorted(S), a, k, m, g, f)].append(a)
    return d

bad_full = bad_onto = tot_full = tot_onto = 0
for k, m in ((2,2), (2,3)):
    for g in maps(k, m, k):
        for f in maps(k, m, 2):
            tot_full += 1
            bad_full += len(classes(4, tuple(range(4)), k, m, g, f)) != 1
            for sz in range(1, 5):
                for S in itertools.combinations(range(4), sz):
                    for p in S:
                        T = tuple(q for q in S if q != p)
                        img = {mn_machine(4, sorted(T), {q: dict(zip(sorted(S), kv))[q]
                                                        for q in T}, k, m, g, f)
                               for kv in itertools.product(range(m), repeat=len(S))}
                        tot_onto += 1
                        bad_onto += img != set(classes(4, T, k, m, g, f).keys())
print(f"  the full cut is a single class      {tot_full:6d} machines        "
      f"{bad_full} exceptions   {'OK' if bad_full == 0 else 'FAIL'}")
print(f"  restriction MN_S -> MN_(S-p) is onto {tot_onto:6d} (cut,position)  "
      f"{bad_onto} exceptions   {'OK' if bad_onto == 0 else 'FAIL'}")

print("\nand the reverse reading -- the profile is pinned at both ends\n")
for n in (3, 4, 5, 6):
    prof = [len(classes(n, tuple(range(t)), 2, 4, g_add, f_add)) for t in range(n + 1)]
    print(f"  adder n={n}: LSB-first class counts {prof}   ends stateless: "
          f"{prof[0] == prof[-1] == 1}")

the two facts the proof actually uses

  the full cut is a single class        4352 machines        0 exceptions   OK
  restriction MN_S -> MN_(S-p) is onto 139264 (cut,position)  0 exceptions   OK

and the reverse reading -- the profile is pinned at both ends

  adder n=3: LSB-first class counts [1, 2, 2, 1]   ends stateless: True
  adder n=4: LSB-first class counts [1, 2, 2, 2, 1]   ends stateless: True
  adder n=5: LSB-first class counts [1, 2, 2, 2, 2, 1]   ends stateless: True
  adder n=6: LSB-first class counts [1, 2, 2, 2, 2, 2, 1]   ends stateless: True




Two things fall out, and both make this notebook claim *less* than its first version did.

**Theorem A′(a) is free.** Forward congruence held in every machine swept, and the reason
needs no machine at all: $\mathrm{MN}_{S\cup\{p\}}(\alpha[p\mapsto u])$ is obtained from
$\mathrm{MN}_S(\alpha)$ by restricting its residual function to the completions with
$p = u$ and dropping whichever coordinates have gone constant. That is an operation on the
residual function itself. So given the theorem of [1] that $\Phi$ *is* the Myhill–Nerode
quotient, congruence follows with no reference to addition. §6 labelled A′(a) VERIFIED
ONLY next to A′(b); it should have said automatic. **The entire content of Theorem A′ is
in (b)** — that $U$ is *bounded*, reading at most two run signatures and stopping.

**Theorem A″ is Proposition 1.** Reverse the reading. Forward, one-wayness looks
manufactured at the cut. Backward, nothing is manufactured anywhere: the class count is
pinned to $1$ at both ends of any traversal — which is §2.1 — so a traversal that starts
at one state and finishes at one state has to rise somewhere, and a rising step has more
classes after than before, so no function inverts it.

$$\text{Directionality is the shape of the ladder, not a property of the step.}$$

Compression is not the cause. The cause is the boundary condition that a complete reading
owes nothing, already stated in §2.1 and used there for something else. A″ and
Proposition 1 are one fact reached by two routes — another address to something already
there, not a new object. Worth saying plainly, because the opposite reading
(irreversibility *generated* by future-sufficient compression) is the more attractive one
and it does not survive being run.

---

## 5. The static side: a predicate with no dynamics

The same machine carries a second structure. On the Cayley graph of $\mathbb{Z}/2^n$ with
generators $2^0,\dots,2^{n-1}$, declare a square commutative when the two moves flip
disjoint sets of bits. That predicate defines a 2-complex, and the residual
$b_1 = \binom{n}{2} + (n-1)$ is the rank the previous paper computed.

Here the question is not how big the residual is. It is whether the predicate's cut is a
*thing* by the same criterion the reader passed. It is not, and the reason is sharper
than a computation.

### 5.1 The predicate is a window predicate

In [34]:
print("declared(x,i,j)  <=>  bits i..j-1 of x are NOT all 1\n")
for n in range(2, 9):
    bad = sum(declared(x, i, j, n) != window_rule(x, i, j, n)
              for x in range(1 << n) for i, j in itertools.combinations(range(n), 2))
    print(f"  n={n}: {bad} disagreements   {'OK' if bad == 0 else 'FAIL'}")

print("\nthe highest bit any window reads")
for n in range(2, 9):
    print(f"  n={n}: max(j-1) over pairs i<j<=n-1  =  {n-2}  =  n-2"
          f"   |   bit n-1 = {n-1} is never read")

declared(x,i,j)  <=>  bits i..j-1 of x are NOT all 1

  n=2: 0 disagreements   OK
  n=3: 0 disagreements   OK
  n=4: 0 disagreements   OK
  n=5: 0 disagreements   OK
  n=6: 0 disagreements   OK
  n=7: 0 disagreements   OK
  n=8: 0 disagreements   OK

the highest bit any window reads
  n=2: max(j-1) over pairs i<j<=n-1  =  0  =  n-2   |   bit n-1 = 1 is never read
  n=3: max(j-1) over pairs i<j<=n-1  =  1  =  n-2   |   bit n-1 = 2 is never read
  n=4: max(j-1) over pairs i<j<=n-1  =  2  =  n-2   |   bit n-1 = 3 is never read
  n=5: max(j-1) over pairs i<j<=n-1  =  3  =  n-2   |   bit n-1 = 4 is never read
  n=6: max(j-1) over pairs i<j<=n-1  =  4  =  n-2   |   bit n-1 = 5 is never read
  n=7: max(j-1) over pairs i<j<=n-1  =  5  =  n-2   |   bit n-1 = 6 is never read
  n=8: max(j-1) over pairs i<j<=n-1  =  6  =  n-2   |   bit n-1 = 7 is never read


### 5.2 The blind bit, and its reverse

Forward reading: the predicate never looks at bit $n-1$, so it has a blind direction.

Reverse reading: the predicate determines everything *else* exactly. The adjacent pairs
alone recover every bit below the top, and every other pair is a disjunction of them.
Both readings are the same fact, and the reverse one is the honest one:

$$\text{the declared set } \textbf{is} \text{ the quotient } \mathbb{Z}/2^n \to \mathbb{Z}/2^{n-1}.$$

The blind bit is not a defect in the predicate. It is the cut the predicate *is*. Placing
the incompleteness in the object would be the mistake here; it is in the reader, and the
reader is the predicate.

In [36]:
print("the predicate factors through Z/2^n -> Z/2^(n-1), exactly\n")
for n in range(2, 9):
    M = 1 << (n - 1)
    pairs = list(itertools.combinations(range(n), 2))
    adj = [(i, i + 1) for i in range(n - 1)]
    cong = all(declared(x, i, j, n) == declared(x + M, i, j, n)
               for x in range(M) for i, j in pairs)
    prof = {}
    for x in range(1 << n):
        prof.setdefault(tuple(declared(x, i, j, n) for i, j in adj), set()).add(x % M)
    sep = len(prof) == M and all(len(v) == 1 for v in prof.values())
    ors = all(declared(x, i, j, n) == any(declared(x, t, t + 1, n) for t in range(i, j))
              for x in range(1 << n) for i, j in pairs)
    print(f"  n={n}: constant on cosets of <2^(n-1)>: {str(cong):>5} | "
          f"the {len(adj)} adjacent predicates separate all {M:>3} residues: {str(sep):>5} | "
          f"all {len(pairs):>2} predicates are ORs of them: {str(ors):>5}   "
          f"{'OK' if cong and sep and ors else 'FAIL'}")

the predicate factors through Z/2^n -> Z/2^(n-1), exactly

  n=2: constant on cosets of <2^(n-1)>:  True | the 1 adjacent predicates separate all   2 residues:  True | all  1 predicates are ORs of them:  True   OK
  n=3: constant on cosets of <2^(n-1)>:  True | the 2 adjacent predicates separate all   4 residues:  True | all  3 predicates are ORs of them:  True   OK
  n=4: constant on cosets of <2^(n-1)>:  True | the 3 adjacent predicates separate all   8 residues:  True | all  6 predicates are ORs of them:  True   OK
  n=5: constant on cosets of <2^(n-1)>:  True | the 4 adjacent predicates separate all  16 residues:  True | all 10 predicates are ORs of them:  True   OK
  n=6: constant on cosets of <2^(n-1)>:  True | the 5 adjacent predicates separate all  32 residues:  True | all 15 predicates are ORs of them:  True   OK
  n=7: constant on cosets of <2^(n-1)>:  True | the 6 adjacent predicates separate all  64 residues:  True | all 21 predicates are ORs of them:  True   OK
  n=8: cons

### 5.3 The stabiliser

A transformation that is to act on the declared complex must preserve the declared set.
Which translations $x \mapsto x+v$ do?

In [38]:
print("translations preserving the declared set\n")
for n in range(2, 9):
    N = 1 << n
    good = stabiliser(n, lambda x, i, j, n=n: declared(x, i, j, n))
    print(f"  n={n}: {good}   = {{0, 2^(n-1)}}   {'OK' if good == [0, 1 << (n-1)] else 'FAIL'}")

translations preserving the declared set

  n=2: [0, 2]   = {0, 2^(n-1)}   OK
  n=3: [0, 4]   = {0, 2^(n-1)}   OK
  n=4: [0, 8]   = {0, 2^(n-1)}   OK
  n=5: [0, 16]   = {0, 2^(n-1)}   OK
  n=6: [0, 32]   = {0, 2^(n-1)}   OK
  n=7: [0, 64]   = {0, 2^(n-1)}   OK
  n=8: [0, 128]   = {0, 2^(n-1)}   OK


The two halves of that, proved for all $n$ rather than observed at small width:

**($\supseteq$)** Adding $2^{n-1}$ changes only bit $n-1$, which no window reads. So it
preserves every declared value.

**($\subseteq$)** Let $v \notin \{0, 2^{n-1}\}$ and let $k$ be its least set bit; then
$k \le n-2$. Take the pair $(i,j) = (k,k+1)$, whose predicate is "bit $k$ of $x$ is not
$1$", and take $x = 0$. Bits $0..k-1$ of $v$ are zero, so adding $v$ produces no carry
into position $k$ and bit $k$ of $x+v$ is $x_k \oplus 1 = 1$. So $(k,k+1)$ is declared at
$x$ and undeclared at $x+v$. $\square$

The witness is uniform in $n$ and needs no search. Here it is, checked against every $v$
at every width.

In [40]:
print("the witness  (i,j) = (k, k+1),  x = 0,  k = least set bit of v\n")
for n in range(2, 10):
    N = 1 << n; bad = 0; tested = 0
    for v in range(1, N):
        if v == 1 << (n - 1): continue
        k = (v & -v).bit_length() - 1
        if k > n - 2: continue
        tested += 1
        if not (declared(0, k, k + 1, n) and not declared(v % N, k, k + 1, n)):
            bad += 1
    print(f"  n={n}: {tested:4d} values of v outside {{0, 2^(n-1)}} | "
          f"{bad} where the stated witness fails   {'OK' if bad == 0 else 'FAIL'}")

the witness  (i,j) = (k, k+1),  x = 0,  k = least set bit of v

  n=2:    2 values of v outside {0, 2^(n-1)} | 0 where the stated witness fails   OK
  n=3:    6 values of v outside {0, 2^(n-1)} | 0 where the stated witness fails   OK
  n=4:   14 values of v outside {0, 2^(n-1)} | 0 where the stated witness fails   OK
  n=5:   30 values of v outside {0, 2^(n-1)} | 0 where the stated witness fails   OK
  n=6:   62 values of v outside {0, 2^(n-1)} | 0 where the stated witness fails   OK
  n=7:  126 values of v outside {0, 2^(n-1)} | 0 where the stated witness fails   OK
  n=8:  254 values of v outside {0, 2^(n-1)} | 0 where the stated witness fails   OK
  n=9:  510 values of v outside {0, 2^(n-1)} | 0 where the stated witness fails   OK


Widening the search does not help. Affine maps $x \mapsto ux+v$, XOR by a constant,
negation, and translations combined with a permutation of the generator labels — all of
them land in the same place.

In [42]:
print("affine maps x -> u*x + v, u a unit\n")
for n in range(2, 8):
    N = 1 << n
    good = sorted((u, v) for u in range(1, N, 2) for v in range(N)
                  if all(declared((u * x + v) % N, i, j, n) == declared(x, i, j, n)
                         for x in range(N) for i, j in itertools.combinations(range(n), 2)))
    pred = sorted({(u % N, v % N) for u in (1, 1 + (1 << (n - 1))) for v in (0, 1 << (n - 1))})
    print(f"  n={n}: {good}   (Z/2)^2 as predicted: {good == pred}")

print("\nnegation x -> -x,  XOR x -> x ^ w\n")
for n in range(2, 9):
    N = 1 << n; pairs = list(itertools.combinations(range(n), 2))
    neg = all(declared((-x) % N, i, j, n) == declared(x, i, j, n)
              for x in range(N) for i, j in pairs)
    xors = [w for w in range(N)
            if all(declared(x ^ w, i, j, n) == declared(x, i, j, n)
                   for x in range(N) for i, j in pairs)]
    print(f"  n={n}: negation preserves: {str(neg):>5} | XOR constants: {xors}")

print("\ntranslations combined with a permutation of the generator labels\n")
for n in (3, 4, 5):
    N = 1 << n; pairs = list(itertools.combinations(range(n), 2))
    good = [(v, pi) for v in range(N) for pi in itertools.permutations(range(n))
            if all(declared((x + v) % N, min(pi[i], pi[j]), max(pi[i], pi[j]), n)
                   == declared(x, i, j, n) for x in range(N) for i, j in pairs)]
    print(f"  n={n}: {len(good)} pairs, all with pi = identity: {good}")

affine maps x -> u*x + v, u a unit

  n=2: [(1, 0), (1, 2), (3, 0), (3, 2)]   (Z/2)^2 as predicted: True
  n=3: [(1, 0), (1, 4), (5, 0), (5, 4)]   (Z/2)^2 as predicted: True
  n=4: [(1, 0), (1, 8), (9, 0), (9, 8)]   (Z/2)^2 as predicted: True
  n=5: [(1, 0), (1, 16), (17, 0), (17, 16)]   (Z/2)^2 as predicted: True
  n=6: [(1, 0), (1, 32), (33, 0), (33, 32)]   (Z/2)^2 as predicted: True
  n=7: [(1, 0), (1, 64), (65, 0), (65, 64)]   (Z/2)^2 as predicted: True

negation x -> -x,  XOR x -> x ^ w

  n=2: negation preserves:  True | XOR constants: [0, 2]
  n=3: negation preserves: False | XOR constants: [0, 4]
  n=4: negation preserves: False | XOR constants: [0, 8]
  n=5: negation preserves: False | XOR constants: [0, 16]
  n=6: negation preserves: False | XOR constants: [0, 32]
  n=7: negation preserves: False | XOR constants: [0, 64]
  n=8: negation preserves: False | XOR constants: [0, 128]

translations combined with a permutation of the generator labels

  n=3: 2 pairs, all with pi = i

Note the $n=2$ line: negation *does* preserve the predicate there, and fails from
$n=3$ on. At $n=2$ the only pair is $(0,1)$, whose predicate reads parity, and negation
preserves parity. The witness that kills it for $n \ge 3$ is $x=3$ with the pair $(0,2)$:
bits $0,1$ are both $1$, so undeclared; $-3 \equiv 1 \pmod 4$ has bit $1$ clear, so
declared.

This corrects a version of the result in circulation which lists negation among the
symmetries. It is not one.

### 5.4 Walking up from behind

Order 4 is not the interesting number, and chasing it is chasing the named object. The
question to ask is what constraint makes the stabiliser what it is. So: widen from *this*
predicate to *all* window predicates — assign to each pair $(i,j)$ an arbitrary set
$W_{ij}$ of allowed patterns on bits $i..j-1$ — and look at the whole family at once.

In [44]:
n = 3
pairs = list(itertools.combinations(range(n), 2))
spaces = [list(itertools.chain.from_iterable(
    itertools.combinations(range(1 << (j - i)), k) for k in range(1 + (1 << (j - i)))))
    for i, j in pairs]
tot = 1
for s in spaces: tot *= len(s)
print(f"all {tot} window predicates at n=3, grouped by (highest bit read, |stabiliser|)\n")

tally = {}
for choice in itertools.product(*spaces):
    W = {pairs[t]: set(choice[t]) for t in range(len(pairs))}
    pr = lambda x, i, j, W=W: ((x >> i) & ((1 << (j - i)) - 1)) in W[(i, j)]
    h = effective_top(n, pr, pairs)
    s = len(stabiliser(n, pr, pairs))
    tally[(h, s)] = tally.get((h, s), 0) + 1

print(f"  {'h':>3} {'|Stab|':>7} {'2^(n-1-h)':>10} {'count':>7}")
for (h, s) in sorted(tally):
    print(f"  {h:>3} {s:>7} {(1 << (n - 1 - h)) if h >= 0 else (1 << n):>10} {tally[(h,s)]:>7}")

all 256 window predicates at n=3, grouped by (highest bit read, |stabiliser|)

    h  |Stab|  2^(n-1-h)   count
   -1       8          8       8
    0       4          4      24
    1       2          2     224


Twenty-four of these are state-dependent predicates whose stabiliser is *bigger* than
$\{0,2^{n-1}\}$ — for instance "declared iff $x_0 = 0$", stabilised by every even
translation. Taken as counterexamples they would kill the claim that state-dependent
predicates cannot be invariant. They are not counterexamples. They are the same law at a
different value of $h$.

**Theorem B′ (stabiliser law).** Let $D$ be any window predicate on $\mathbb{Z}/2^n$ and
let $h$ be its *effective top bit*: the largest $t$ with $D_{ij}(x \oplus 2^t) \ne
D_{ij}(x)$ for some $x$ and pair $(i,j)$, or $-1$ if there is none. Then

$$\mathrm{Stab}(D) \;=\; \langle 2^{h+1}\rangle \;=\; \{v : v \equiv 0 \bmod 2^{h+1}\}.$$

*Proof.* $\mathrm{Stab}(D)$ is a subgroup of $\mathbb{Z}/2^n$, and every subgroup of
$\mathbb{Z}/2^n$ is $\langle 2^m \rangle$ for a unique $m$. So it is enough to locate $m$.

$m \le h+1$: if $v \equiv 0 \bmod 2^{h+1}$ then $v$ has no set bit at or below $h$, so
adding $v$ generates no carry into positions $0..h$ and fixes bits $0..h$ of $x$. No
$D_{ij}$ depends on a bit above $h$. So $2^{h+1} \in \mathrm{Stab}(D)$.

$m \ge h+1$: suppose $h \ge 0$. By definition of $h$ there are $x$ and $(i,j)$ with
$D_{ij}(x \oplus 2^h) \ne D_{ij}(x)$. Now $x + 2^h$ and $x \oplus 2^h$ agree on bits
$0..h$ — adding $2^h$ leaves bits below $h$ alone, complements bit $h$, and carries only
upward — and $D_{ij}$ reads no bit above $h$. So $D_{ij}(x + 2^h) = D_{ij}(x \oplus 2^h)
\ne D_{ij}(x)$, and $2^h \notin \mathrm{Stab}(D)$. $\square$

The step that makes this work is the one that looked like the obstacle. Attacking an
arbitrary $v$ requires routing the carry from its least set bit up to a bit the predicate
actually consults, and for a predicate that reads bit $h$ while skipping bits below it
there is no clean route. Passing to the subgroup removes the need for any such route:
only the single element $2^h$ has to be excluded. The obstacle was an artifact of chasing
the named object instead of the constraint that pins it.

An earlier version of this notebook reported the $\subseteq$ half as verified and claimed
the consequence below "holds either way". That was a non-sequitur — the consequence is
logically equivalent to $\subseteq$ — and noticing it is what produced the proof.

$$\text{every translation that preserves a declared set acts trivially on every bit that set reads.}$$

There is no non-vacuous translation symmetry to be had, for any window predicate, at any
width. The adder's predicate is the $h = n-2$ case; its stabiliser is small because it
reads almost everything.

In [46]:
print("each step of the proof, checked before it was written down\n")

def windows3():
    n = 3; pairs = list(itertools.combinations(range(n), 2))
    spaces = [list(itertools.chain.from_iterable(
        itertools.combinations(range(1 << (j - i)), k) for k in range(1 + (1 << (j - i)))))
        for i, j in pairs]
    for choice in itertools.product(*spaces):
        W = {pairs[t]: set(choice[t]) for t in range(len(pairs))}
        yield (lambda x, i, j, W=W: ((x >> i) & ((1 << (j - i)) - 1)) in W[(i, j)]), pairs

def sample(n, k, seed):
    random.seed(seed); prs = list(itertools.combinations(range(n), 2)); out = []
    for _ in range(k):
        W = {}
        for (i, j) in prs:
            w = j - i
            if random.random() < 0.5:                       # arbitrary window
                W[(i, j)] = {p for p in range(1 << w) if random.random() < 0.5}
            else:                                            # reads ONE bit, skipping below
                t = random.randrange(w)
                W[(i, j)] = {p for p in range(1 << w) if (p >> t) & 1}
        out.append((lambda x, i, j, W=W: ((x >> i) & ((1 << (j-i)) - 1)) in W[(i, j)], prs))
    return out

# step 0: Stab is a subgroup, and therefore of the form <2^m>
nsub = ncyc = 0; cases3 = list(windows3())
for pr, prs in cases3:
    N = 8; S = set(stabiliser(3, pr, prs))
    nsub += not (0 in S and all((a + b) % N in S for a in S for b in S))
    m = min((v & -v).bit_length() - 1 for v in S if v) if S != {0} else 3
    ncyc += S != (set(range(0, N, 1 << m)) if m < 3 else {0})
print(f"  step 0  Stab is a subgroup of the form <2^m>   n=3: {nsub} not closed, "
      f"{ncyc} not cyclic of that form   {'OK' if nsub == ncyc == 0 else 'FAIL'}")

# steps 2 and 3, and the conclusion
for n, cases in [(3, cases3), (4, sample(4, 400, 2026)), (5, sample(5, 400, 2026)),
                 (6, sample(6, 400, 2026))]:
    N = 1 << n; bad_up = bad_dn = bad_all = nowit = 0
    for pr, prs in cases:
        h = effective_top(n, pr, prs)
        g = (1 << (h + 1)) % N
        bad_up += not all(pr((x + g) % N, i, j) == pr(x, i, j)
                          for x in range(N) for i, j in prs)
        if h >= 0:
            wit = next(((x, i, j) for x in range(N) for i, j in prs
                        if pr(x ^ (1 << h), i, j) != pr(x, i, j)), None)
            if wit is None: nowit += 1
            else:
                x, i, j = wit
                bad_dn += pr((x + (1 << h)) % N, i, j) == pr(x, i, j)
        want = list(range(0, N, 1 << (h + 1))) if h >= 0 else list(range(N))
        bad_all += sorted(stabiliser(n, pr, prs)) != want
    tag = "all 256" if n == 3 else "400 sampled, half reading one bit and skipping below"
    print(f"  n={n} ({tag})")
    print(f"     step 2   2^(h+1) in Stab           {bad_up} exceptions   "
          f"{'OK' if bad_up == 0 else 'FAIL'}")
    print(f"     step 3   2^h not in Stab, via the witness  {bad_dn} exceptions "
          f"({nowit} with no witness)   {'OK' if bad_dn == nowit == 0 else 'FAIL'}")
    print(f"     concl.   Stab = <2^(h+1)>          {bad_all} violations   "
          f"{'OK' if bad_all == 0 else 'FAIL'}")

each step of the proof, checked before it was written down

  step 0  Stab is a subgroup of the form <2^m>   n=3: 0 not closed, 0 not cyclic of that form   OK
  n=3 (all 256)
     step 2   2^(h+1) in Stab           0 exceptions   OK
     step 3   2^h not in Stab, via the witness  0 exceptions (0 with no witness)   OK
     concl.   Stab = <2^(h+1)>          0 violations   OK
  n=4 (400 sampled, half reading one bit and skipping below)
     step 2   2^(h+1) in Stab           0 exceptions   OK
     step 3   2^h not in Stab, via the witness  0 exceptions (0 with no witness)   OK
     concl.   Stab = <2^(h+1)>          0 violations   OK
  n=5 (400 sampled, half reading one bit and skipping below)
     step 2   2^(h+1) in Stab           0 exceptions   OK
     step 3   2^h not in Stab, via the witness  0 exceptions (0 with no witness)   OK
     concl.   Stab = <2^(h+1)>          0 violations   OK
  n=6 (400 sampled, half reading one bit and skipping below)
     step 2   2^(h+1) in Stab       

  n=4 (400 sampled, half reading one bit and skipping below)
     step 2   2^(h+1) in Stab           0 exceptions   OK
     step 3   2^h not in Stab, via the witness  0 exceptions (0 with no witness)   OK
     concl.   Stab = <2^(h+1)>          0 violations   OK


  n=5 (400 sampled, half reading one bit and skipping below)
     step 2   2^(h+1) in Stab           0 exceptions   OK
     step 3   2^h not in Stab, via the witness  0 exceptions (0 with no witness)   OK
     concl.   Stab = <2^(h+1)>          0 violations   OK


  n=6 (400 sampled, half reading one bit and skipping below)
     step 2   2^(h+1) in Stab           0 exceptions   OK
     step 3   2^h not in Stab, via the witness  0 exceptions (0 with no witness)   OK
     concl.   Stab = <2^(h+1)>          0 violations   OK


### 5.6 The pointwise stabiliser

The six families above are not six results. **Proposition 8.** A map
$\sigma : \mathbb{Z}/2^n \to \mathbb{Z}/2^n$ preserves every declared value if and only if
$\sigma(x) \equiv x \bmod 2^{n-1}$; the bijections among these form a group of order
$2^{2^{n-1}}$.

*Proof.* If $\sigma(x) \equiv x \bmod 2^{n-1}$ they agree on every bit any window reads.
Conversely the $n-1$ adjacent predicates at $x$ determine $x \bmod 2^{n-1}$ (§5.2), so
agreement of all declared values forces $\sigma(x) \equiv x \bmod 2^{n-1}$. Such a
$\sigma$ is $x \mapsto x + \varepsilon(x)2^{n-1}$, a bijection exactly when $\varepsilon$
is constant on each of the $2^{n-1}$ cosets $\{y, y+2^{n-1}\}$. $\square$

Every family tested above is a restriction of this, which is why they all land in the same
place. Order 4 is not the interesting number, and neither is $2^{2^{n-1}}$.

In [48]:
print("Proposition 8, checked directly\n")
for n in (2, 3, 4, 5, 6):
    N, M = 1 << n, 1 << (n - 1)
    prs = list(itertools.combinations(range(n), 2))
    ok = True; size = 1
    for x in range(N):
        img = [y for y in range(N)
               if all(declared(y, i, j, n) == declared(x, i, j, n) for i, j in prs)]
        if sorted(img) != sorted({x % M, x % M + M}): ok = False
        size *= len(img)
    print(f"  n={n}: every x moves only inside its coset: {str(ok):>5} | "
          f"all such maps 2^{N} = {size} | bijections 2^{M} = {1 << M}   "
          f"{'OK' if ok else 'FAIL'}")

Proposition 8, checked directly

  n=2: every x moves only inside its coset:  True | all such maps 2^4 = 16 | bijections 2^2 = 4   OK
  n=3: every x moves only inside its coset:  True | all such maps 2^8 = 256 | bijections 2^4 = 16   OK
  n=4: every x moves only inside its coset:  True | all such maps 2^16 = 65536 | bijections 2^8 = 256   OK
  n=5: every x moves only inside its coset:  True | all such maps 2^32 = 4294967296 | bijections 2^16 = 65536   OK
  n=6: every x moves only inside its coset:  True | all such maps 2^64 = 18446744073709551616 | bijections 2^32 = 4294967296   OK


### 5.5 The arena's own dynamics

The complex is not an abstract setting with arbitrary candidate transformations. It has
one intrinsic dynamics: its own generators, $x \mapsto x + 2^i$. Those are the moves the
predicate exists to talk about. So ask which of them preserve the declared set.

In [50]:
print("generators x -> x + 2^i preserving the declared set\n")
for n in range(3, 11):
    prs = list(itertools.combinations(range(n), 2))
    good = [i for i in range(n)
            if all(declared((x + (1 << i)) % (1 << n), a, b, n) == declared(x, a, b, n)
                   for x in range(1 << n) for a, b in prs)]
    tors = [i for i in range(n) if (2 * (1 << i)) % (1 << n) == 0]
    print(f"  n={n}: preserving: i in {str(good):<5} | 2-torsion generators: {str(tors):<5} | "
          f"highest bit read: {n-2}   {'same bit' if good == tors == [n-1] else 'DIFFER'}")

generators x -> x + 2^i preserving the declared set

  n=3: preserving: i in [2]   | 2-torsion generators: [2]   | highest bit read: 1   same bit
  n=4: preserving: i in [3]   | 2-torsion generators: [3]   | highest bit read: 2   same bit
  n=5: preserving: i in [4]   | 2-torsion generators: [4]   | highest bit read: 3   same bit
  n=6: preserving: i in [5]   | 2-torsion generators: [5]   | highest bit read: 4   same bit
  n=7: preserving: i in [6]   | 2-torsion generators: [6]   | highest bit read: 5   same bit
  n=8: preserving: i in [7]   | 2-torsion generators: [7]   | highest bit read: 6   same bit
  n=9: preserving: i in [8]   | 2-torsion generators: [8]   | highest bit read: 7   same bit
  n=10: preserving: i in [9]   | 2-torsion generators: [9]   | highest bit read: 8   same bit


One generator, the top one, and it is the same bit as everything else that is
distinguished in this problem: $2^{n-1}$ is the unique involution among the generators, it
is the base case of the induction in the previous paper, it carries the sign
$(-1)^{x_{n-1}}$ in the normal form, and it is the bit the predicate cannot see.

This looked like a coincidence and it is not one — but it is also not four facts. It is
one fact counted four times. In $\mathbb{Z}/2^n$ with generators $2^0..2^{n-1}$, the top
generator is the unique element of order 2 because $2 \cdot 2^{n-1} \equiv 0$; and bit
$n-1$ is unread because a window is $[i,j-1]$ with $j \le n-1$. Both are consequences of
the index range, not of anything about addition. The base case and the sign then follow
from those two in the published argument. Naming the coincidence and stopping there would
have been the error; the constraint that forces it is the content.

So the single generator that preserves the declared set is the single generator that
cannot move anything the predicate reads. The predicate's cut admits no dynamics that
does not act trivially on it. **The circle does not close on this side.**

---

## 6. The criterion

Put the two sides next to each other. Both are cuts of the same machine. One passes, one
fails, and the difference is not a matter of degree.

A cut makes a thing when four conditions hold:

1. **Boundary.** There is a cut — a set, not a history. Its route is erased; only its
   geometry survives. (§1: $F(S)$ mentions only the maximal runs of $S$.)
2. **Compression.** The cut induces a quotient that keeps exactly the distinctions the
   future can use — no fewer, or the state is not sufficient; no more, or it is not a
   state but a record. (§1: $|\mathrm{im}\,\Phi| = |\mathrm{MN}| = F(S)$.)
3. **Non-vacuous dynamics.** There is a transition that *moves what the quotient
   distinguishes* — not merely one the quotient survives.
4. **Bounded update.** The induced transition reads a fixed amount of the quotient plus its
   local coupling, not the whole quotient and not the history it summarises. (§3: Thm A′(b).)

Earlier versions had a single third clause, "the next state is a function of the current
state and the local coupling". That is forward congruence, which §4.3 shows is automatic:
**5,648 of 5,648 readers satisfy it, and 1,812 of those have no state at all.** A clause
every candidate passes is not a clause. What is left after removing it is the two halves
that do separate things.

The reader's cut satisfies all four. The predicate's cut satisfies the first two —
$K_D$ is a boundary and $B_A/B_D$ is a quotient — and fails clause 3, not marginally but
structurally: by Theorem B′ the only translations preserving it act trivially on
everything it reads, and that holds for every window predicate, not just this one. A
*stateless* reader fails clause 3 too, for the opposite reason — its dynamics is fine, but
its quotient distinguishes nothing for the dynamics to act on. One clause, two ways to
fail it, and this machine exhibits both.

The two verdicts carry different weights and it matters which. The **pass** rests on
Theorem A′(b), verified to $n=6$ and not proved — A′(a) is automatic (§4.2) and carries
nothing. The **failure** rests on Theorem B′ and Proposition 7, both proved for all $n$. So
the criterion is currently better at ruling things out than at ruling them in, which is the
honest way round for a test.

To be explicit about what "fails" means here, it is worth checking that the failure is a
fact about *this* predicate and not about the arena. Compare three predicates on the same
complex.

In [53]:
print(f"{'n':>2}  {'predicate':<9} {'#declared':>10} {'|Stab|':>7} {'b1':>6}  "
      f"{'state-dependent?':>17}")
for n in range(3, 8):
    for name, pr in (("none",     lambda x, i, j: False),
                     ("disjoint", lambda x, i, j, n=n: declared(x, i, j, n)),
                     ("all",      lambda x, i, j: True)):
        b, cnt = b1_of(n, pr)
        st = stabiliser(n, pr)
        dep = any(pr(x, i, j) != pr(0, i, j)
                  for x in range(1 << n) for i, j in itertools.combinations(range(n), 2))
        print(f"{n:>2}  {name:<9} {cnt:>10} {len(st):>7} {b:>6}  {str(dep):>17}")
    print()

 n  predicate  #declared  |Stab|     b1   state-dependent?
 3  none               0       8     13              False
 3  disjoint          14       2      5               True
 3  all               24       8      2              False

 4  none               0      16     41              False
 4  disjoint          62       2      9               True
 4  all               96      16      3              False

 5  none               0      32    113              False
 5  disjoint         222       2     14               True
 5  all              320      32      4              False

 6  none               0      64    289              False
 6  disjoint         702       2     20               True
 6  all              960      64      5              False

 7  none               0     128    705              False
 7  disjoint        2046       2     27               True
 7  all             2688     128      6              False



 7  none               0     128    705              False


 7  disjoint        2046       2     27               True


 7  all             2688     128      6              False



The two unconditional predicates are invariant under every translation — trivially,
since they do not consult the state at all. Only the state-dependent one has a small
stabiliser. So the collapse is not an artifact of the arena.

But note what the table does *not* show: stabiliser size does not track residual size.
`none` and `all` have identical stabilisers and residuals at opposite ends of the range.
Whatever the relationship between symmetry and residual is, it is not monotone, and this
paper does not claim one.

### 6.1 The full set

Two cuts in this machine have $F = 1$: the empty one and the complete one. Neither is an
object, and for opposite reasons. The empty cut has drawn no distinction yet. The complete
cut has no continuation left in which a distinction could matter. In between, and only in
between, there is something to retain.

That is the precise version of *the cut is what makes the thing exist*. Not that the cut
reveals a thing that was already separate, and not that the whole is incomplete. The whole
is complete; $F = 1$ says so. It is complete in a way that leaves it no local address from
which to act on itself as one thing among others. An entity is a recurrence across a cut
*within* a whole that lacks nothing.

The shape of an entity is then the triple

$$\text{entity} \;=\; (\text{cut},\ \text{continuation quotient},\ \textbf{bounded }\text{local update law}),$$

and its identity is the persistence of the triple, not of anything inside the boundary.
The machine gives this content rather than metaphor: the cut is $S$, the quotient is
$\Phi_S$, the update law is $U$, and Theorem A′(b) is the statement that the third exists
*and is bounded*. The word doing the work is **bounded** — drop it and the third slot is
filled by every quotient there is.

### 6.2 What this does and does not license

The criterion is a **test**, not a derivation. Nothing here derives physics, and nothing
here shows that physical objects satisfy it. What the machine supplies is the demonstration
that the test is not vacuous — a single system carrying one cut that passes and one that
fails — together with a worked case in which the *failure* is a theorem rather than an
impression. The pass is a finite-width verification, and §7 keeps that distinction.

Stated as a test: given a candidate entity, exhibit a coarse-graining $\Phi$ of its states
and a transition $F$ with $\Phi \circ F = \bar{F} \circ \Phi$, where $F$ *moves what $\Phi$
distinguishes* and $\bar{F}$ reads a *bounded* amount of $\Phi$ plus local coupling. The
commuting square on its own is free (§4.3), so the test is carried entirely by those two
side conditions.

One caution the machine itself supplies. §4 shows the update is irreversible: a state
determines its successor and not its predecessor. So "dynamically closed" in this sense
does not mean reversible, and a criterion built on congruence will not by itself recover a
time-symmetric dynamics. Theorem A‴ sharpens that: under (H1) and (H2) a reader quotient
that *is* reversible is one with no state at all, so the criterion cannot be strengthened
to demand both. But §4.2 shows the escape — break (H2) and reversible non-trivial
quotients exist. A limitation to state plainly, not a feature.

---

## 7. Status

What is proved for all $n$, what is verified at finite width, and what is neither.

In [55]:
rows = [
 ("cut law F(S) = prod f(R)",                 "proved (prev. paper)", "n <= 6 exhaustive"),
 ("Phi is exactly the MN quotient",           "proved (prev. paper)", "n <= 6 exhaustive"),
 ("Prop 1  F=1 iff S empty or S=[0,n-1]",     "proved (this paper)",  "n <= 8 exhaustive"),
 ("Prop 2  F=2 iff S=[0,r] or S={n-1}",       "proved (this paper)",  "n <= 8 exhaustive"),
 ("Cor 3   {n-1} is a dead end",              "proved (this paper)",  "n <= 8 exhaustive"),
 ("Cor 4   LSB-first unique, both costs",     "proved (this paper)",  "n <= 8, both functionals"),
 ("Prop 5  interior run => not a power of 2", "proved (this paper)",  "n = 4..8 exhaustive"),
 ("Thm A'(a)  Phi is a congruence",           "AUTOMATIC from Phi=MN","52,304 transducers, 0 fails"),
 ("Thm A'(b)  U is local",                    "VERIFIED ONLY",        "n = 3..6; thin at merge->int"),
 ("Thm A''    un-reading is not a function",  "proved; = Prop 1",     "n = 3..6; witness all n>=3"),
 ("Thm A3'    backward-func iff stateless",   "proved, any (H1)+(H2)","52,304 transducers, 0 mismatch"),
 ("   4.2   (H1) automatic, (H2) load-bearing","proved / witnessed",   "240 quotients; open vs closed"),
 ("   6     clause 3 as written is vacuous",   "shown (this paper)",   "5,648 of 5,648 readers pass"),
 ("   2.4   cut law holds in every base",      "proved / verified",    "b=2..5 every cut, 0 exceptions"),
 ("   2.4   f_bot = 2 in every base",          "proved (this paper)",  "carry is binary whatever b"),
 ("   2.4   f_top=(b-1)L+1, f_int=2(b-1)L+1",  "VERIFIED ONLY",        "b=2..5, 64 single-run cuts"),
 ("Prop 6  declared = window predicate",      "proved (this paper)",  "n <= 8 exhaustive"),
 ("        factors through Z/2^(n-1), exact", "proved (this paper)",  "n <= 8 exhaustive"),
 ("Prop 7  Stab(adder) = {0, 2^(n-1)}",       "proved (this paper)",  "n <= 8 exhaustive"),
 ("        negation fails for n >= 3",        "proved (this paper)",  "n <= 8 exhaustive"),
 ("Prop 8  pointwise Stab, order 2^(2^(n-1))","proved (this paper)",  "n <= 6 exhaustive"),
 ("Thm B'  Stab(D) = <2^(h+1)>, all windows", "proved (this paper)",  "n=3 all 256; n=4,5,6 sampled"),
 ("        only 2^(n-1) preserves K_D",       "proved (this paper)",  "n <= 10 exhaustive"),
]
w = max(len(r[0]) for r in rows)
print(f"{'claim':<{w}}  {'status':<21}  evidence")
print("-" * (w + 2 + 21 + 2 + 29))
for a, b, c in rows:
    print(f"{a:<{w}}  {b:<21}  {c}")
print()
print("There is now exactly ONE verified-only entry: Thm A'(b), the locality of U.")
print("A'(a) is automatic for any exact MN quotient of any machine, so it follows from")
print("the previous paper with no adder in it.  A'' is Proposition 1 by a second route")
print("(section 4.2).  Everything downstream of A'(b) -- the 'passes' verdict of section")
print("6 -- inherits its strength.  The 'fails' verdict does not: it rests on B' and 7.")

claim                                       status                 evidence
------------------------------------------------------------------------------------------------
cut law F(S) = prod f(R)                    proved (prev. paper)   n <= 6 exhaustive
Phi is exactly the MN quotient              proved (prev. paper)   n <= 6 exhaustive
Prop 1  F=1 iff S empty or S=[0,n-1]        proved (this paper)    n <= 8 exhaustive
Prop 2  F=2 iff S=[0,r] or S={n-1}          proved (this paper)    n <= 8 exhaustive
Cor 3   {n-1} is a dead end                 proved (this paper)    n <= 8 exhaustive
Cor 4   LSB-first unique, both costs        proved (this paper)    n <= 8, both functionals
Prop 5  interior run => not a power of 2    proved (this paper)    n = 4..8 exhaustive
Thm A'(a)  Phi is a congruence              AUTOMATIC from Phi=MN  52,304 transducers, 0 fails
Thm A'(b)  U is local                       VERIFIED ONLY          n = 3..6; thin at merge->int
Thm A''    un-reading is not a f

### Open

1. **Theorem A′(b) for all $n$.** Prove the locality of $U$ by induction on the run
   decomposition, rather than by exhaustion to $n=6$. Only (b) is left: (a) is automatic
   (§4.2). The thinnest cell in the verification is `merge → interior`, so that is where an
   induction should start.

2. **Beyond translations, and beyond $\mathbb{Z}/2^n$.** Theorem B′ closes the
   translation question completely, and negatively. Its proof uses two properties of the
   setting that will not transfer: that the subgroups of $\mathbb{Z}/2^n$ form a chain, so
   $\mathrm{Stab}$ is generated by one element, and that carries propagate only upward, so
   a window has a well-defined top bit. On a group whose subgroup lattice is not a chain
   the argument has no first step. Which groups and generating sets admit a window-like
   predicate with a non-vacuous stabiliser is the concrete form of problem 3.

3. **Dynamical admissibility of a relation predicate.** *(This replaces the open problem
   about which physical process selects the predicate, which was the wrong question.)*
   Characterise the pairs $(D, T)$ where $D$ is a relation predicate on a Cayley complex
   and $T$ a transformation with $T(K_D) = K_D$, such that $T$ acts **non-trivially on the
   state $D$ reads**. Then $T$ descends to $B_A/B_D$ and the static residual acquires
   dynamics. The adder is a negative example, and by Theorem B′ the obstruction is not
   special to it: for every window predicate on $\mathbb{Z}/2^n$, a translation that fixes
   a low-bit window is a translation that does not move it. The whole family is ruled out
   at once. Does the obstruction survive a change of group, or of generating set? A
   positive example would be worth more than another negative one, and after Theorem B′ it
   will have to come from outside $\mathbb{Z}/2^n$.

4. ~~**Is the cut ever two-way?**~~ **Closed in §4.1.** Theorem A‴ answers it: un-reading
   is a function exactly when the machine is stateless, for any transducer. The problem's
   own second clause was wrong — it guessed irreversibility would turn out to be "a
   property of compression rather than of the adder". It is neither. The proof uses only
   that the full cut is stateless and that restriction is onto, so the one-wayness is a
   property of the ladder's endpoints, and Proposition 1 already said it. The replacement
   question is narrower and worth asking: those two hypotheses are weak enough to hold
   well outside finite transducers, so which coarse-graining set-ups violate one of them,
   and does a reversible non-trivial quotient live there?

5. ~~**$\sum \log_2 F = n-1$ and $b_1(\text{all squares}) = n-1$.**~~ **Closed, negatively,
   in §2.4.** The settlement named was: deform the machine and see whether both $n-1$s move
   together. Changing base is that deformation, and they do not. The reader-side $n-1$ is
   base-invariant. The static-side count is $n-1$ only at $b=2$ and is $n$ for every
   $b \ge 3$ — because at $b=2$ the top generator is an involution and removes one
   dimension, while for $b \ge 3$ nothing is. The agreement was a coincidence whose cause
   is the same 2-torsion that carries the blind bit, the base case and the sign. A fourth
   thing was hanging on that bit; now it is off.

---

## 8. Corrections made while writing

Four passes were made over this material, the last an adversarial read by a referee with
no knowledge of how any of it was arrived at. Each pass broke something an earlier one had
asserted. They are listed because the null results were navigational, not because the
record is tidy.

| # | asserted | what running it showed |
|---|---|---|
| 1 | $F(S)=1$ iff $S=[0,n-1]$ | false — the empty cut has $F=1$ too. Became §2.1, the clearest statement of the thesis. |
| 2 | every pair predicate is an OR of the adjacent ones | true, but first coded with a negation in the wrong place and reported as FAIL. The bug was mine; the claim survived. |
| 3 | state-dependent predicates cannot have a large stabiliser | false — 24 of the 256 window predicates at $n=3$ do. Forced the move from cataloguing automorphism families to asking what fixes the stabiliser. |
| 4 | $\mathrm{Aut}$ includes negation (from a review) | false for $n \ge 3$; true only at $n=2$. Corrected rather than adopted. |
| 5 | the update has four geometry cases | three. "Both neighbours in the same run" cannot occur, since $p \notin S$ separates them. The missing twelfth cell is `open → full`, a (geometry, run type) pair rather than a geometry. |
| 6 | the $\subseteq$ half is open but "the consequence holds either way" | incoherent — the consequence is logically *equivalent* to $\subseteq$. Looking for which of the two to keep produced the subgroup argument, and §5.4 is now a theorem for all $n$. |
| 7 | Theorem A′ is "what this paper proves" | contradicted the status table in the same document, which lists both halves VERIFIED ONLY. All such sentences now say verified. |
| 8 | Theorem A′(a) is a result of this notebook about the adder | it is automatic for any exact MN quotient of any machine, so it follows from [1] with no adder in it. 52,304 transducers, 0 forward failures. The content of A′ is entirely in (b). |
| 9 | Theorem A″ shows compression generates irreversibility | direction right, cause wrong. A‴ proves the dichotomy for any transducer from two facts — full cut stateless, restriction onto — so it is the shape of the ladder, and §4.2 shows A″ is Proposition 1 by a second route. |

Rows 6 and 7 came from the referee. Rows 8 and 9 came from running an arrow this notebook
had drawn rather than agreeing with it, and both make it claim less than it did. Row 7 is
the failure mode this programme warns about most directly — a status table kept honest
while the prose around it quietly promotes the result — and it survived three of my own
passes.

---

## 9. Reproduction

Run this notebook top to bottom from a clean kernel. It needs `numpy` and nothing else.
Every table in the paper is the printed output of the cell above it. No result in the paper
is quoted from a previous session.

In [57]:
import platform, sys
print("python   ", sys.version.split()[0], "on", platform.system())
print("numpy    ", np.__version__)
print()
print("If every cell above printed OK and no FAIL, the paper's claims at the stated")
print("widths have been reproduced on this machine.")

python    3.12.7 on Windows
numpy     1.26.4

If every cell above printed OK and no FAIL, the paper's claims at the stated
widths have been reproduced on this machine.
